In [ ]:
import json
import pandas as pd
import os
from pathlib import Path

# Run this notebook from within the 09_llm_categorization/ directory.
# All intermediate JSONL files are saved here; input/output CSVs use relative paths.
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

data = pd.read_csv(project_root / "data" / "raw" / "literature_dataset.csv")
dataShort = data

In [ ]:
# chunk not needed if you already have a parsed column

def parse_keywords(row):
    # Get Title and Abstract, treating NaN as an empty string
    parsed_input = (row['Title'] if pd.notna(row['Title']) else '') + " " + (row['Abstract'] if pd.notna(row['Abstract']) else '')
    
    # Check for keywords in the specified order and add the first non-empty one found
    if pd.notna(row['Author Keywords']) and row['Author Keywords'].strip():
        parsed_input += " " + row['Author Keywords']
    elif pd.notna(row['WOS Keywords']) and row['WOS Keywords'].strip():
        parsed_input += " " + row['WOS Keywords']
    elif pd.notna(row['Scopus Keywords']) and row['Scopus Keywords'].strip():
        parsed_input += " " + row['Scopus Keywords']

    return parsed_input

# Apply the function row by row
# dataShort['parsedInput'] = dataShort.apply(parse_keywords, axis=1)

# def parse_keywords(row):
#     # Get Title and Abstract, treating NaN as an empty string
#     parsed_input = (row['Title'] if pd.notna(row['Title']) else '') + " " + (row['Abstract'] if pd.notna(row['Abstract']) else '')
    
#     # Check for keywords in the specified order and add the first non-empty one found
#     if pd.notna(row['Author Keywords']) and row['Author Keywords'].strip():
#         parsed_input += " " + row['Author Keywords']
#     elif pd.notna(row['Scopus Keywords']) and row['Scopus Keywords'].strip():
#         parsed_input += " " + row['Scopus Keywords']
#     elif pd.notna(row['WOS Keywords']) and row['WOS Keywords'].strip():
#         parsed_input += " " + row['WOS Keywords']
    
#     return parsed_input

# Apply the function row by row
dataShort['parsedInput'] = dataShort.apply(parse_keywords, axis=1)

In [ ]:
# List to hold all API call objects
batch_requests = []
for idx, parsedText in enumerate(dataShort['parsedInput'], start=1):
    batch_requests.append({
        "custom_id": f"{dataShort['ID'][idx-1]}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # "model": "gpt-5-mini", # choose one of the two
            "model": "o4-mini", # choose one of the two
            "reasoning_effort": "medium", # that's default
            "n": 3, #
            "temperature": 1, # would do smaller for non-reasoning models, but reasoning models are 1 by default
            "messages": 
                [{"role": "developer", "content": "You are an expert at reviewing papers in the social science related to air pollution."},
                 {"role": "user", "content": f"""
             This is the title, abstract, and keywords for a given research paper: "```{parsedText}```"
             From the mention of a country, city , a national organization or the general context, identify every single country that is mentioned in the study.
             The mention of a country related to copyright or a publisher should not be considered.

             The paper can cover one or multiple countries. If you cannot find information about any country, please return None as a country.
             
             Steps to follow:
             - write a small explanation of why you would label a paper with (a) specific country name(s). 
             - write "output_json".
             - Then make a new line and only write, in a JSON format between curly brackets, the output where the key is "countries" and the value is a list of countries such as:
                {{
                    "countries": ["CountryName1", "CountryName2", ...]
                }}
             """
             }],
            # "max_tokens": 300 # if using gpt-4.1-mini or similar
            'max_completion_tokens': 800, # if using o4-mini or similar reasoning model
        }
    })

with open("batch_requests_country_o4mini_final.jsonl", "w", encoding="utf-8") as fout:
    for req in batch_requests:
        fout.write(json.dumps(req) + "\n")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="INSERT-YOUR-KEY-HERE")

In [ ]:
batch_input_file = client.files.create(
    file=open("batch_requests_country_o4mini_final.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

In [ ]:
batch_input_file_id = batch_input_file.id
batch_created = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "country papers o4mini n5 fullCall",
    }
)

print(batch_created)
print(batch_created.id)
batch_id = batch_created.id

In [ ]:
# 2) Retrieve the batch by its ID
# batch = client.batches.retrieve('batch_68b6c33acf64819090012028331a7f74')
batch = client.batches.retrieve(batch_id)

# 3) Print the full Batch object
print(batch)

# extract the batch ID and output file id from the response
batch_id = batch.id
output_file_id = batch.output_file_id

error_file_id = batch.error_file_id


In [ ]:
# check for error
file_error = client.files.content(error_file_id) # latest batch_68b6c33acf64819090012028331a7f74
file_error.text

In [ ]:
file_response = client.files.content(output_file_id) # latest batch_68b6c33acf64819090012028331a7f74
print(file_response.text)

file_response.write_to_file("batch_requests_country_o4mini_final_response.jsonl")

In [ ]:
# import json
# import pandas as pd

# 1. Read your JSONL
with open("batch_requests_country_o4mini_final_response.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

# 2. Normalize, exploding the choices list into rows
df = pd.json_normalize(
    records,
    record_path=["response", "body", "choices"],
    meta=[
        "custom_id",
        ["response", "body", "model"],
        ["response", "body", "created"],
        ["id"]
    ],
    errors="ignore"
)

# 4. Clean up column names if you like
df = df.rename(
    columns={
        "index": "choice_index",
        "message.content": "message_content"
    }
)

print(df)

In [ ]:
import ast

def _first_balanced_brace_block(text: str) -> str | None:
    """Return the first {...} block with balanced braces, ignoring braces inside strings."""
    if not text:
        return None
    depth = 0
    start = None
    in_str = False
    quote = None
    escape = False

    for i, ch in enumerate(text):
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == quote:
                in_str = False
            continue

        if ch in ("'", '"'):
            in_str = True
            quote = ch
            continue

        if ch == '{':
            if depth == 0:
                start = i
            depth += 1
        elif ch == '}':
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    return text[start:i+1]
    return None

def extract_identification(text: str):
    # 1) find a balanced JSON-like object
    obj = _first_balanced_brace_block(text)
    if not obj:
        return None

    # 2) parse: strict JSON first, then tolerant Python-literal fallback
    try:
        data = json.loads(obj)
    except json.JSONDecodeError:
        try:
            data = ast.literal_eval(obj)  # handles single quotes, True/False/None, trailing commas
        except Exception:
            return None

    countries = data.get("countries")
    if countries is None:
        return None
    if isinstance(countries, (list, tuple)):
        # return a clean comma-separated string
        return ", ".join(str(c) for c in countries)
    return str(countries)

# usage
df["countriesChatGPT"] = df["message_content"].apply(extract_identification)
# no extra cleaning step needed, but if you want only letters/commas/spaces:
df["countriesChatGPTClean"] = df["countriesChatGPT"].str.replace(r"[^A-Za-zÀ-ÿ ,\-]", "", regex=True)

In [ ]:
# select important columns and save to csv
df_save = df[['custom_id', "choice_index", 'message_content', 'countriesChatGPTClean', 'response.body.model']]

df_save.to_csv(project_root / "data" / "llm_classifications" / "countries_o4mini.csv", index=False)